In [ ]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(1)

producer.flush()
producer.close()

In [ ]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
from datetime import datetime, timedelta
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na anomalie prędkości...")

user_timestamps = {}

for message in consumer:
    tx = message.value
    user_id = tx['user_id']
    now = datetime.fromisoformat(tx['timestamp'])

    if user_id not in user_timestamps:
        user_timestamps[user_id] = []
    user_timestamps[user_id].append(now)

    nowa_lista = []
    
    for t in user_timestamps[user_id]:
        if now - t < timedelta(seconds=60):
            nowa_lista.append(t)
    user_timestamps[user_id] = nowa_lista

    if len(user_timestamps[user_id]) > 3:
        print(f"ALERT: {user_id} | {len(user_timestamps[user_id])} transakcji w 60s")